In [33]:
# Import Basic Packgaes 
import numpy as np
import pandas as pd
import datetime
import statsmodels as sm
import itertools
import glob
import os
from scipy.signal import argrelextrema
from scipy.signal import find_peaks, peak_widths

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns # advanced vizs
from pandas.plotting import lag_plot

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Set Color Palettes
palette1 = itertools.cycle(sns.color_palette(palette='Set1'))
palette2 = itertools.cycle(sns.color_palette(palette='Set2'))

In [34]:
os.chdir("D:\\TDMS\\14.P3_Validation-4\\P3_Validation-4_FFT")

In [35]:
os.getcwd()

'D:\\TDMS\\14.P3_Validation-4\\P3_Validation-4_FFT'

In [36]:
# current directory csv files
csvs = [x for x in os.listdir("D:\\TDMS\\14.P3_Validation-4\\P3_Validation-4_FFT") if x.endswith('.csv')]
# stats.csv -> stats
fns = [os.path.splitext(os.path.basename(x))[0] for x in csvs]

dic_csv = {}
for i in range(len(fns)):
    dic_csv[fns[i]] = pd.read_csv(csvs[i])

In [37]:
dic_csv.keys()

dict_keys(['9.P3_Validation-4_15Nov23'])

In [38]:
#dic_csv['11.P3_Validation-4_17Nov23'].head()

In [39]:
for key, df in dic_csv.items():
    df['Time'] =pd.to_datetime(df['Time'])

In [40]:
for key, df in dic_csv.items():
    df.set_index(['Time'], inplace=True)

In [41]:
'''
a = len(dic_csv)  # number of rows
b = 1  # number of columns
c = 1  # initialize plot counter

fig1 = plt.figure(figsize=(30,90))

SMALL_SIZE = 16

for j,i in resample_dic_90ms.items():
    plt.subplot(a, b, c)
    #plt.vlines(x=i.index, ymin=0, ymax=i["Cycle-detect"])
    #plt.plot(i.index, i['P1-Water Discharge Pressure'])
    #plt.plot(i.index, i['P2-Air Supply Flowrate'])
    #plt.plot(i.index, i['P1-Air-Supply-pressure'])
    #plt.plot(i.index, i['P2-Water Suction Pressure'])
    #plt.plot(i.index, i['P2-Water Discharge Flowrate'])
    #plt.plot(i.index, i['CPM_diff'])
    plt.plot(i.index, i['P2-CPM'])
    plt.ylim(0,80)
    plt.title(j)
    c = c + 1

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=SMALL_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.show()
'''

'\na = len(dic_csv)  # number of rows\nb = 1  # number of columns\nc = 1  # initialize plot counter\n\nfig1 = plt.figure(figsize=(30,90))\n\nSMALL_SIZE = 16\n\nfor j,i in resample_dic_90ms.items():\n    plt.subplot(a, b, c)\n    #plt.vlines(x=i.index, ymin=0, ymax=i["Cycle-detect"])\n    #plt.plot(i.index, i[\'P1-Water Discharge Pressure\'])\n    #plt.plot(i.index, i[\'P2-Air Supply Flowrate\'])\n    #plt.plot(i.index, i[\'P1-Air-Supply-pressure\'])\n    #plt.plot(i.index, i[\'P2-Water Suction Pressure\'])\n    #plt.plot(i.index, i[\'P2-Water Discharge Flowrate\'])\n    #plt.plot(i.index, i[\'CPM_diff\'])\n    plt.plot(i.index, i[\'P2-CPM\'])\n    plt.ylim(0,80)\n    plt.title(j)\n    c = c + 1\n\nplt.rc(\'font\', size=SMALL_SIZE)          # controls default text sizes\nplt.rc(\'axes\', titlesize=SMALL_SIZE)     # fontsize of the axes title\nplt.rc(\'axes\', labelsize=SMALL_SIZE)    # fontsize of the x and y labels\nplt.rc(\'xtick\', labelsize=SMALL_SIZE)    # fontsize of the tick labe

In [42]:
# Define FFT parameters
dt = 0.03
f1, f2 = 1000, 1000  # Hz
diff = 100

In [43]:
for key, df in dic_csv.items():
    # Define the time interval for slicing
    start_time = pd.to_datetime('09:00:00.000')
    end_time = pd.to_datetime('17:46:59.999')
    time_interval = pd.Timedelta(minutes=2)
    
    current_time = start_time
    while current_time <= end_time:
        next_time = current_time + time_interval

        # Slice the DataFrame based on the specified time range
        sliced_df = df.between_time(current_time.time(), next_time.time())

        # Perform FFT analysis only if there is data in the sliced DataFrame
        n = len(sliced_df)
        if n > 0:
            t = sliced_df.index
            s = sliced_df['P3-Air Suction Pressure']

            # FFT
            s -= s.mean()
            fr = np.fft.rfftfreq(n, dt)
            fou = np.fft.rfft(s)

            # Plot and save the FFT plot
            plt.figure(figsize=(28, 10))
            plt.subplot(211)
            plt.plot(t, s)
            plt.title('Data with Noise')

            plt.subplot(212)
            plt.plot(np.fft.fftshift(fr), np.fft.fftshift(np.abs(fou)))
            plt.xticks(np.arange(min(np.fft.fftshift(fr)), max(np.fft.fftshift(fr)), 0.3))
            plt.ylim(0, 6000)
            plt.title('Spectrum of Noisy Data')

            # Save the plot with a meaningful filename
            filename = os.path.join("D:\\TDMS\\14.P3_Validation-4\\P3_Validation-4_FFT\\11.P3_Validation-4_15Nov23_2minFFT", f'{key}_{current_time.strftime("%H-%M-%S")}_{next_time.strftime("%H-%M-%S")}_FFT.png')
            plt.savefig(filename)
            plt.close()
            #fr_list = [np.fft.fftshift(fr)]
            #amp_list = [np.fft.fftshift(np.abs(fou))]
            #fourier_df = pd.DataFrame(data=[fr_list, amp_list],columns=['frequency','amplidute'])
            #print(fourier_df.loc[fourier_df['amplidute'].idxmax()])

        else:
            # Handle the case where the DataFrame is empty (e.g., print a message)
            print(f"No data in the time range {current_time} - {next_time}")

        current_time = next_time

No data in the time range 2023-12-01 09:00:00 - 2023-12-01 09:02:00
No data in the time range 2023-12-01 17:46:00 - 2023-12-01 17:48:00
